# Cronbach's Alpha Reliability Analysis

This notebook computes Cronbach's alpha for the Likert-scale constructs used in the blended learning thesis project.

The analysis is configuration-driven. Dataset paths, output files, construct definitions, overall Likert items, reverse-coding rules, interpretation thresholds, and alpha-if-item-deleted settings are loaded from:

```text
ml/config/config.json -> reliability
```

This reduces notebook-level errors when the cleaned dataset location, item list, or construct definitions change.


## 1. Import libraries and load configuration

In [1]:
from pathlib import Path
import sys

# Make imports work from either project/ml or project/ml/notebook.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    src_dir = candidate / "src"
    config_file = candidate / "config" / "config.json"
    if src_dir.exists() and config_file.exists():
        ML_ROOT = candidate.resolve()
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise FileNotFoundError("Could not find the ml project root containing config/config.json and src/.")

import re
import numpy as np
import pandas as pd

from blended_learning.config.settings import settings
from blended_learning.analysis.reliability import (
    alpha_if_item_deleted,
    cronbach_alpha,
    interpret_alpha,
    slugify_filename,
    slugify_construct_name,
    validate_item_columns,
)

reliability_cfg = settings.reliability
io_cfg = reliability_cfg["io"]

input_path_key = io_cfg.get("input_path_key", "output_cleaned_path")
DATA_PATH = Path(settings.path[input_path_key])
OUTPUT_DIR = (ML_ROOT / io_cfg["output_dir"]).resolve()
READ_CSV_OPTIONS = io_cfg.get("read_csv_options", {})
WRITE_CSV_OPTIONS = io_cfg.get("write_csv_options", {"index": False, "encoding": "utf-8-sig"})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ML root:", ML_ROOT)
print("Config file:", settings.config_path)
print("Dataset path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)


ML root: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml
Config file: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\config\config.json
Dataset path: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\cleaned_data.csv
Output directory: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability


## 2. Define helper functions

In [2]:
# Reliability helper functions are imported from blended_learning.analysis.reliability.
# Keeping them in src avoids copy-paste across notebooks and scripts.


## 3. Load the cleaned dataset

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check reliability.io.input_path_key in config/config.json."
    )

df = pd.read_csv(DATA_PATH, **READ_CSV_OPTIONS)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (592, 87)


,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,career_preparation,ideal_balance,prefer_more_blended,open_strengths,open_challenges_suggestions,survey_start,survey_end,response_time_minutes,student_id,flag_speeder
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,2,"More Online than Face-to-Face (e.g., 40% In-pe...",No,NaN,NaN,2026-03-12 00:35:35.641,2026-03-12 00:37:43.468,2.130450,e20210528,True
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,3,"More Online than Face-to-Face (e.g., 40% In-pe...",Neutral/Unsure,Nothing,Nothing,2026-03-17 18:53:40.435,2026-03-17 18:56:16.257,2.597033,e20210686,True
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,4,"Balanced Half and Half (50% In-person, 50% Onl...",Neutral/Unsure,"Very good, excellent","No big challenge, i’m the best",2026-03-09 15:33:21.321,2026-03-09 15:36:31.387,3.167767,e20241146,False
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,3,"Mostly Face-to-Face (e.g., 80% In-person, 20% ...",Neutral/Unsure,Will try hard,Lack of self-discipline,2026-03-09 15:32:15.225,2026-03-09 15:35:36.121,3.348267,e20240609,False
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,3,"Balanced Half and Half (50% In-person, 50% Onl...",Neutral/Unsure,getting more experience,Discipline on daily studying,2026-03-09 15:31:35.190,2026-03-09 15:35:12.175,3.616417,e20240542,False


## 4. Load Likert-scale constructs from config

The construct definitions and the overall Likert item pool are maintained in `config.json` under `reliability`. The item `tech_issues_freq` is reverse-coded according to the `reverse_coding` rule before calculating the overall reliability because higher original values indicate more frequent technical issues.


In [4]:
constructs = reliability_cfg["constructs"]
all_likert_items = reliability_cfg["overall_likert_items"]
overall_construct_name = reliability_cfg["overall_construct_name"]
reverse_coding_cfg = reliability_cfg.get("reverse_coding", {})
alpha_thresholds = reliability_cfg["alpha_interpretation_thresholds"]

print("Number of constructs:", len(constructs))
print("Number of construct-level items:", sum(len(items) for items in constructs.values()))
print("Number of overall Likert items:", len(all_likert_items))
print("Reverse-coded items:", list(reverse_coding_cfg.keys()))


Number of constructs: 8
Number of construct-level items: 32
Number of overall Likert items: 33
Reverse-coded items: ['tech_issues_freq']


## 5. Check that all required columns exist

In [5]:
required_columns = set(all_likert_items)

for items in constructs.values():
    required_columns.update(items)

missing_columns = sorted([col for col in required_columns if col not in df.columns])

if missing_columns:
    raise ValueError(
        "The following required columns are missing from the cleaned dataset:\n"
        + "\n".join(missing_columns)
    )

print("All required Likert-scale columns are available.")


All required Likert-scale columns are available.


## 6. Apply reverse-coding rules

Reverse-coding rules are loaded from `reliability.reverse_coding`. For the current survey, `tech_issues_freq` is reverse-coded because higher original values mean more frequent technical issues, while most other Likert items are positively oriented.


In [6]:
df_alpha = df.copy()
all_likert_items_for_alpha = list(all_likert_items)

for source_col, rule in reverse_coding_cfg.items():
    output_col = rule["output_column"]
    min_value = rule.get("min_value", 1)
    max_value = rule.get("max_value", 5)

    df_alpha[source_col] = pd.to_numeric(df_alpha[source_col], errors="coerce")
    df_alpha[output_col] = (min_value + max_value) - df_alpha[source_col]

    all_likert_items_for_alpha = [
        output_col if col == source_col else col
        for col in all_likert_items_for_alpha
    ]

preview_cols = []
for source_col, rule in reverse_coding_cfg.items():
    preview_cols.extend([source_col, rule["output_column"]])

df_alpha[preview_cols].head() if preview_cols else pd.DataFrame()


,tech_issues_freq,tech_issues_freq_reversed
0,2,4
1,3,3
2,3,3
3,4,2
4,3,3


## 7. Compute Cronbach's alpha

In [7]:
results = []

# Overall reliability
overall_alpha, overall_n, overall_k = cronbach_alpha(
    df_alpha[all_likert_items_for_alpha]
)

results.append(
    {
        "Construct": overall_construct_name,
        "Number of Items": overall_k,
        "Valid Responses": overall_n,
        "Cronbach Alpha": round(overall_alpha, 3)
        if not pd.isna(overall_alpha)
        else np.nan,
        "Interpretation": interpret_alpha(overall_alpha, alpha_thresholds),
    }
)

# Construct-level reliability
for construct_name, items in constructs.items():
    construct_data = df[items]
    alpha, n_valid, k_items = cronbach_alpha(construct_data)

    results.append(
        {
            "Construct": construct_name,
            "Number of Items": k_items,
            "Valid Responses": n_valid,
            "Cronbach Alpha": round(alpha, 3)
            if not pd.isna(alpha)
            else np.nan,
            "Interpretation": interpret_alpha(alpha, alpha_thresholds),
        }
    )

results_df = pd.DataFrame(results)
results_df


,Construct,Number of Items,Valid Responses,Cronbach Alpha,Interpretation
0,Overall Likert-Scale Item Pool,33,592,0.881,Good
1,Learning Material Use,6,592,0.623,Questionable / moderate
2,Engagement and Interaction,4,592,0.568,Low / questionable
3,Course Integration and Learning Understanding,2,592,0.467,Low / questionable
4,Lecturer Support,5,592,0.814,Good
5,Self-Regulation,4,592,0.605,Questionable / moderate
6,Perceived Benefits,6,592,0.772,Acceptable
7,Digital Learning Readiness and Usability,3,592,0.495,Low / questionable
8,Learning Outcome and Future Readiness,2,592,0.661,Questionable / moderate


In [8]:
construct_item_count = sum(len(items) for items in constructs.values())

print("Number of constructs:", len(constructs))
print("Number of construct-level items:", construct_item_count)
print("Number of overall Likert items:", len(all_likert_items))
print("Items not included in constructs:")

construct_items = set(item for items in constructs.values() for item in items)
overall_items = set(all_likert_items)

missing_from_constructs = sorted(overall_items - construct_items)
print(missing_from_constructs)


Number of constructs: 8
Number of construct-level items: 32
Number of overall Likert items: 33
Items not included in constructs:
['tech_issues_freq']


The 33 ordinal Likert-scale variables were grouped into eight multi-item constructs for construct-level reliability analysis. The variable \texttt{tech\_issues\_freq} was treated as a single technical issues indicator and was therefore not assessed as a separate construct using Cronbach's alpha. However, it was reverse-coded and included in the overall 33-item reliability analysis.

## 8. Compute alpha-if-item-deleted diagnostics

## What “Alpha if Item Deleted” Means

The alpha-if-item-deleted analysis is a diagnostic step used to examine whether each questionnaire item contributes positively to the internal consistency of a scale or construct. It recalculates Cronbach's alpha after removing one item at a time.

If the alpha value becomes higher after deleting an item, it may indicate that the item is less consistent with the other items in the same construct. If the alpha value becomes lower or stays almost the same, the item does not appear to reduce the reliability of the construct.

In this study, alpha-if-item-deleted was used only as a diagnostic check. Items were not automatically removed based only on this result because the questionnaire variables were selected based on their relevance to blended learning behavior and perception. Therefore, item removal was considered only when both statistical evidence and theoretical meaning supported it.


### How to Interpret Alpha-if-Item-Deleted

The alpha-if-item-deleted result shows what Cronbach's alpha would become if one item was removed from the item pool or construct. It is used as a diagnostic check to see whether any item reduces internal consistency.

| Case | Meaning |
|---|---|
| Alpha if item deleted is lower than original alpha | Keep the item |
| Alpha if item deleted is much higher than original alpha | The item may not fit well |
| Alpha if item deleted is almost the same | The item is not harmful |

In this study, alpha-if-item-deleted was used only as a diagnostic tool. Items were not removed automatically because each variable was selected based on its relevance to blended learning behavior and perception.

In [9]:
# =====================================================
# ALPHA-IF-ITEM-DELETED DIAGNOSTICS
# =====================================================

alpha_deleted_cfg = reliability_cfg["alpha_if_item_deleted"]
alpha_deleted_files = {}

if alpha_deleted_cfg.get("enabled", True):
    # 1. Overall alpha-if-item-deleted
    overall_deleted = alpha_if_item_deleted(
        df_alpha[all_likert_items_for_alpha]
    )

    overall_deleted_path = OUTPUT_DIR / alpha_deleted_cfg["overall_filename"]
    overall_deleted.to_csv(overall_deleted_path, **WRITE_CSV_OPTIONS)
    alpha_deleted_files[overall_construct_name] = overall_deleted_path.name

    print(f"Saved overall alpha-if-item-deleted diagnostics: {overall_deleted_path}")

    # 2. Construct-level alpha-if-item-deleted
    skipped_constructs = []
    min_items = alpha_deleted_cfg.get("minimum_items", 3)
    prefix = alpha_deleted_cfg["construct_filename_prefix"]

    for construct_name, items in constructs.items():
        item_count = len(items)

        if item_count < min_items:
            skipped_constructs.append(
                {
                    "Construct": construct_name,
                    "Number of Items": item_count,
                    "Reason": f"Skipped because alpha-if-item-deleted is not meaningful for constructs with fewer than {min_items} items.",
                }
            )
            continue

        construct_data = df[items]
        deleted_result = alpha_if_item_deleted(construct_data)
        safe_name = slugify_construct_name(construct_name)
        file_name = f"{prefix}{safe_name}.csv"
        output_path = OUTPUT_DIR / file_name

        deleted_result.to_csv(output_path, **WRITE_CSV_OPTIONS)
        alpha_deleted_files[construct_name] = file_name

        print(f"Saved construct alpha-if-item-deleted diagnostics: {output_path}")

    # 3. Save skipped construct note
    if skipped_constructs:
        skipped_df = pd.DataFrame(skipped_constructs)
        skipped_path = OUTPUT_DIR / alpha_deleted_cfg["skipped_constructs_filename"]
        skipped_df.to_csv(skipped_path, **WRITE_CSV_OPTIONS)
        print(f"Saved skipped construct note: {skipped_path}")

    print("\nAlpha-if-item-deleted diagnostics completed.")
else:
    overall_deleted = pd.DataFrame()
    print("Alpha-if-item-deleted diagnostics disabled in config.json.")

overall_deleted.head()


Saved overall alpha-if-item-deleted diagnostics: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_overall_33_items.csv
Saved construct alpha-if-item-deleted diagnostics: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_learning_material_use.csv
Saved construct alpha-if-item-deleted diagnostics: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_engagement_and_interaction.csv
Saved construct alpha-if-item-deleted diagnostics: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_lecturer_support.csv
Saved construct alpha-if-item-deleted diagnostics: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_self_regulation.csv
Saved construct alpha-if-ite

,Deleted Item,Remaining Items,Valid Responses,Alpha if Item Deleted
0,video_helpfulness,32,592,0.879
1,digital_literacy_improvement,32,592,0.876
2,use_lecture_slides,32,592,0.878
3,use_video_lectures,32,592,0.878
4,use_quizzes,32,592,0.879


Alpha-if-item-deleted diagnostics were computed for the overall 33-item Likert-scale pool and for constructs containing at least three items. Two-item constructs were not included in the item-deletion diagnostics because removing one item would leave only a single item, making Cronbach's alpha no longer meaningful.

## 9. Save reliability results for thesis appendix

In [10]:
outputs_cfg = reliability_cfg["outputs"]
latex_cfg = reliability_cfg["latex"]

results_csv_path = OUTPUT_DIR / outputs_cfg["results_csv"]
results_tex_path = OUTPUT_DIR / outputs_cfg["results_latex"]

results_df.to_csv(results_csv_path, **WRITE_CSV_OPTIONS)

latex_table = results_df.to_latex(
    index=False,
    escape=latex_cfg.get("escape", False),
    caption=latex_cfg["caption"],
    label=latex_cfg["label"],
    column_format=latex_cfg["column_format"],
    float_format=latex_cfg["float_format"],
)

with open(results_tex_path, "w", encoding="utf-8") as f:
    f.write(latex_table)

print("Saved output files:")
print(f"- {results_csv_path}")
print(f"- {results_tex_path}")
if alpha_deleted_cfg.get("enabled", True):
    print(f"- {OUTPUT_DIR / alpha_deleted_cfg['overall_filename']}")


Saved output files:
- C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\cronbach_alpha_results.csv
- C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\cronbach_alpha_table.tex
- C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_overall_33_items.csv


## Thesis Interpretation Template

Use the generated `results_df` and `alpha_deleted_interpretation_df` tables to update the methodology, appendix, or results discussion. Avoid hardcoding sample size or alpha values in the notebook text because they can change when the cleaned dataset changes.

Suggested interpretation pattern:

```text
To assess the reliability of the Likert-scale items used in the survey, Cronbach's alpha was computed using the final cleaned dataset. The reliability analysis was conducted after preprocessing, using the ordinal Likert-scale variables defined in the project configuration file. Negatively oriented items were reverse-coded before computing the overall item-pool reliability.

The overall Cronbach's alpha for the Likert-scale item pool was [insert alpha from cronbach_alpha_results.csv], indicating [insert interpretation]. Construct-level reliability was also examined because the questionnaire covered multiple dimensions of blended learning experience. Constructs with lower alpha values were interpreted cautiously because the questionnaire was designed as an exploratory perception survey rather than a fully validated psychometric scale.

Alpha-if-item-deleted diagnostics were used to check whether removing any item meaningfully improved reliability. Items were retained unless the increase was meaningful and theoretically justified.
```


In [11]:
# =====================================================
# INTERPRET ALPHA-IF-ITEM-DELETED RESULTS
# =====================================================

interpretation_results = []
decision_thresholds = alpha_deleted_cfg["decision_thresholds"]

for _, row in results_df.iterrows():
    construct_name = row["Construct"]
    original_alpha = row["Cronbach Alpha"]

    file_name = alpha_deleted_files.get(construct_name)

    if file_name is None:
        interpretation_results.append(
            {
                "Construct": construct_name,
                "Original Alpha": original_alpha,
                "Highest Alpha if Item Deleted": None,
                "Item with Highest Deleted Alpha": None,
                "Difference": None,
                "Decision": "No alpha-if-item-deleted file was generated for this construct.",
            }
        )
        continue

    file_path = OUTPUT_DIR / file_name

    if not file_path.exists():
        interpretation_results.append(
            {
                "Construct": construct_name,
                "Original Alpha": original_alpha,
                "Highest Alpha if Item Deleted": None,
                "Item with Highest Deleted Alpha": None,
                "Difference": None,
                "Decision": "No alpha-if-item-deleted file found.",
            }
        )
        continue

    deleted_df = pd.read_csv(file_path)

    if deleted_df["Alpha if Item Deleted"].isna().all():
        interpretation_results.append(
            {
                "Construct": construct_name,
                "Original Alpha": original_alpha,
                "Highest Alpha if Item Deleted": None,
                "Item with Highest Deleted Alpha": None,
                "Difference": None,
                "Decision": "Skip interpretation because deleting one item leaves too few items for Cronbach's alpha.",
            }
        )
        continue

    max_idx = deleted_df["Alpha if Item Deleted"].idxmax()
    highest_alpha = deleted_df.loc[max_idx, "Alpha if Item Deleted"]
    item_name = deleted_df.loc[max_idx, "Deleted Item"]
    difference = highest_alpha - original_alpha

    if difference > decision_thresholds["clear_improvement"]:
        decision = "Review this item because deleting it clearly improves alpha."
    elif difference > decision_thresholds["small_increase"]:
        decision = "Small increase only. Keep the item unless there is a strong theoretical reason to remove it."
    elif difference > decision_thresholds["very_small_increase"]:
        decision = "Very small increase. Keep the item."
    else:
        decision = "Keep all items because deleting any item does not improve alpha."

    interpretation_results.append(
        {
            "Construct": construct_name,
            "Original Alpha": original_alpha,
            "Highest Alpha if Item Deleted": highest_alpha,
            "Item with Highest Deleted Alpha": item_name,
            "Difference": round(difference, 3),
            "Decision": decision,
        }
    )

alpha_deleted_interpretation_df = pd.DataFrame(interpretation_results)
interpretation_path = OUTPUT_DIR / alpha_deleted_cfg["interpretation_filename"]
alpha_deleted_interpretation_df.to_csv(interpretation_path, **WRITE_CSV_OPTIONS)
print(f"Saved alpha-if-item-deleted interpretation: {interpretation_path}")
alpha_deleted_interpretation_df


Saved alpha-if-item-deleted interpretation: C:\Users\Tepy\Documents\tepy\Final Internship Docs\Blended-Learning\ml\data\processed\reliability\alpha_if_item_deleted_interpretation.csv


,Construct,Original Alpha,Highest Alpha if Item Deleted,Item with Highest Deleted Alpha,Difference,Decision
0,Overall Likert-Scale Item Pool,0.881,0.886,tech_issues_freq_reversed,0.005,Small increase only. Keep the item unless ther...
1,Learning Material Use,0.623,0.604,use_lecture_slides,-0.019,Keep all items because deleting any item does ...
2,Engagement and Interaction,0.568,0.509,comfort_asking_questions,-0.059,Keep all items because deleting any item does ...
3,Course Integration and Learning Understanding,0.467,NaN,None,NaN,No alpha-if-item-deleted file was generated fo...
4,Lecturer Support,0.814,0.805,lect_timely_feedback,-0.009,Keep all items because deleting any item does ...
5,Self-Regulation,0.605,0.572,self_prioritize_deadlines,-0.033,Keep all items because deleting any item does ...
6,Perceived Benefits,0.772,0.754,benefit_recorded_access,-0.018,Keep all items because deleting any item does ...
7,Digital Learning Readiness and Usability,0.495,0.462,lms_usability,-0.033,Keep all items because deleting any item does ...
8,Learning Outcome and Future Readiness,0.661,NaN,None,NaN,No alpha-if-item-deleted file was generated fo...


The alpha-if-item-deleted results are now interpreted from config-driven thresholds. Review `alpha_if_item_deleted_interpretation.csv` and use the `Decision` column as a guide. Retain items unless deleting an item meaningfully improves reliability and there is a strong theoretical reason to remove it.
